# 06 · Unified Namespace (UNS)

**Objetivo:** entender a ideia do UNS: **um broker como fonte única da verdade**, organizado pela hierarquia ISA-95 (`empresa/site/área/linha/célula/dado`), onde produtores publicam o estado atual e consumidores pegam a fatia que interessa com wildcards.

O problema que ele resolve: em vez de N sistemas ligados ponto a ponto (N² integrações), todo mundo fala com o broker.

In [1]:
# Configuração: deixa a raiz do repositório no caminho de import
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from common.notebook import load, run_in_background, require_broker
require_broker()
uns = load("06-unified-namespace/uns.py")

broker MQTT encontrado em localhost:1883


## 1. Os produtores e os consumidores

* **Gateway de CLP (OT):** publica `state` e `count/good` de 4 máquinas, retidos.
* **Conector do MES (TI):** publica a ordem de produção de cada linha, ao lado dos dados das máquinas.
* **4 consumidores**, cada um com um filtro diferente.

In [2]:
for nome, filtro in uns.CONSUMERS:
    print(f"{nome:<28} assina  {uns.ENTERPRISE}/{filtro}")

Plant manager São Paulo      assina  acme-bd6a/sao-paulo/#
OEE app, all fillers         assina  acme-bd6a/+/+/+/filler/#
Andon board, all states      assina  acme-bd6a/+/+/+/+/state
Production planning          assina  acme-bd6a/+/+/+/mes/work-order


In [3]:
resultado = uns.main()

UNS root: acme-bd6a/   (11 topics in the namespace)

Plant manager São Paulo  subscribes 'sao-paulo/#'  -> 8 topics
     sao-paulo/packaging/line-1/capper/count/good 23
     sao-paulo/packaging/line-1/capper/state      FAULTED
     sao-paulo/packaging/line-1/filler/count/good 18
     sao-paulo/packaging/line-1/filler/state      RUNNING
     sao-paulo/packaging/line-1/mes/work-order    WO-9121
     sao-paulo/packaging/line-2/filler/count/good 11
     ... 2 more

OEE app, all fillers  subscribes '+/+/+/filler/#'  -> 4 topics
     sao-paulo/packaging/line-1/filler/count/good 29
     sao-paulo/packaging/line-1/filler/state      RUNNING
     sao-paulo/packaging/line-2/filler/count/good 19
     sao-paulo/packaging/line-2/filler/state      STOPPED

Andon board, all states  subscribes '+/+/+/+/state'  -> 4 topics
     curitiba/molding/line-7/press-01/state       RUNNING
     sao-paulo/packaging/line-1/capper/state      STOPPED
     sao-paulo/packaging/line-1/filler/state      RUNNING
     sao-

**Interpretação, consumidor por consumidor**

* **Gerente de São Paulo** (`sao-paulo/#`): `#` pega tudo abaixo do site: máquinas e ordens das duas linhas de São Paulo, nada de Curitiba.
* **App de OEE** (`+/+/+/filler/#`): cada `+` pula exatamente um nível (site, área, linha). Resultado: **todas as enchedoras de todos os sites**, sem saber quantas existem.
* **Painel Andon** (`+/+/+/+/state`): só os estados, de qualquer máquina.
* **Planejamento** (`+/+/+/mes/work-order`): só as ordens de produção, publicadas pelo MES no mesmo lugar da árvore onde estão as máquinas.

Nenhum consumidor conhece o IP do CLP ou o banco do MES. **O contrato é a estrutura de tópicos.**

## 2. Visualizando a árvore

In [4]:
from collections import defaultdict
arvore = lambda: defaultdict(arvore)
raiz = arvore()
for t in resultado["all"]:
    no = raiz
    for parte in t.split("/"):
        no = no[parte]

def imprimir(no, nivel=0):
    for k in sorted(no):
        print("   " * nivel + ("└─ " if nivel else "") + k)
        imprimir(no[k], nivel + 1)

print(uns.ENTERPRISE)
imprimir(raiz)

acme-bd6a
curitiba
   └─ molding
      └─ line-7
         └─ mes
            └─ work-order
         └─ press-01
            └─ count
               └─ good
            └─ state
sao-paulo
   └─ packaging
      └─ line-1
         └─ capper
            └─ count
               └─ good
            └─ state
         └─ filler
            └─ count
               └─ good
            └─ state
         └─ mes
            └─ work-order
      └─ line-2
         └─ filler
            └─ count
               └─ good
            └─ state
         └─ mes
            └─ work-order


**Interpretação:** é exatamente a hierarquia ISA-95 (enterprise > site > area > line > cell). Dado de OT (`state`, `count/good`) e de TI (`mes/work-order`) convivem no mesmo nível de linha, o que permite responder "quantas peças a linha 1 fez da ordem atual" com uma única assinatura.

## Resumo

* UNS não é um produto, é um **padrão de arquitetura**: hierarquia + estado atual retido + publicação por exceção.
* `+` = um nível, `#` = tudo abaixo.
* O valor está no desacoplamento: adicionar um consumidor novo (IA, dashboard, historiador) não exige mexer em nenhum produtor.
* Limitação do MQTT puro: o payload é livre. O Sparkplug B (notebook 07) padroniza payload, tipos e estado.